<a href="https://colab.research.google.com/github/Mona-Alkhathlan/hello-world/blob/master/AI_Agent_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **GymGenie AI** - AI Gym Assistant

GymGenie AI is a smart member support solution that combines a policy-aware knowledge base assistant with an intelligent service agent. It answers questions from gym policies and operational documents using retrieval-augmented generation (RAG), while also handling member issues through guided agentic workflows such as membership freeze, payment disputes, booking issues, access problems, and cancellation requests.

# **Core features**
### **For the RAG assistant**
* document-based answers
* policy and procedure retrieval
* source-based responses
* branch-specific answers
* multilingual support if needed
### **For the service agent**
* intent detection
* guided issue resolution
* state machine / workflow handling
* rule validation
* escalation to human support
* status confirmation

In [14]:
# Install packages

!pip -q install -U langchain langgraph langsmith langchain-openai langchain-openrouter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 12.7 MB/s eta 0:00:00


"""

Service Agentic Workflow

It uses:
- LangChain agent
- middleware-based step control
- state machine pattern
- tool-based state updates
- in-memory checkpointing

Use case:
A gym customer service assistant that handles issues step by step.


"""

In [18]:
import os
from typing import Callable, Literal, Optional
from typing_extensions import NotRequired

from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
    SummarizationMiddleware,
)
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI # Added this import


# -----------------------------------------------------------------------------
# Model
# -----------------------------------------------------------------------------
# Set your OpenAI API key here or as an environment variable
#api_key=os.environ.get("OPENROUTER_API_KEY")
#model = init_chat_model("openrouter/openai:gpt-4.1-mini", api_key=api_key, model_provider="openrouter")

from google.colab import userdata
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# -----------------------------------------------------------------------------
# Define workflow steps
# -----------------------------------------------------------------------------
GymSupportStep = Literal[
    "issue_classifier",
    "customer_data_collector",
    "issue_resolution_specialist",
]


# -----------------------------------------------------------------------------
# State
# -----------------------------------------------------------------------------
class GymSupportState(AgentState):
    """State for gym customer service workflow."""

    current_step: NotRequired[GymSupportStep]

    # classified issue
    issue_type: NotRequired[
        Literal[
            "freeze_membership",
            "double_charge",
            "qr_access_issue",
            "class_booking_issue",
            "locker_code_issue",
            "unknown",
        ]
    ]

    # customer profile / collected data
    member_id: NotRequired[str]
    customer_name: NotRequired[str]
    branch: NotRequired[str]
    membership_status: NotRequired[Literal["active", "inactive", "frozen", "unknown"]]

    # issue-specific data
    freeze_start_date: NotRequired[str]
    freeze_duration_days: NotRequired[int]

    payment_reference: NotRequired[str]
    payment_date: NotRequired[str]

    class_name: NotRequired[str]
    class_date: NotRequired[str]

    locker_number: NotRequired[str]


# -----------------------------------------------------------------------------
# Tools that update state
# -----------------------------------------------------------------------------
@tool
def record_issue_type(
    issue_type: Literal[
        "freeze_membership",
        "double_charge",
        "qr_access_issue",
        "class_booking_issue",
        "locker_code_issue",
        "unknown",
    ],
    runtime: ToolRuntime[None, GymSupportState],
) -> Command:
    """Record the detected issue type and move to customer data collection."""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"Issue type recorded as: {issue_type}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "issue_type": issue_type,
            "current_step": "customer_data_collector",
        }
    )


@tool
def collect_customer_data(
    runtime: ToolRuntime[None, GymSupportState],
    member_id: Optional[str] = None,
    customer_name: Optional[str] = None,
    branch: Optional[str] = None,
    membership_status: Optional[Literal["active", "inactive", "frozen", "unknown"]] = None,
    freeze_start_date: Optional[str] = None,
    freeze_duration_days: Optional[int] = None,
    payment_reference: Optional[str] = None,
    payment_date: Optional[str] = None,
    class_name: Optional[str] = None,
    class_date: Optional[str] = None,
    locker_number: Optional[str] = None,
) -> Command:
    """Collect customer information needed for the current issue and move to resolution."""

    updates = {
        "current_step": "issue_resolution_specialist",
        "messages": [
            ToolMessage(
                content="Customer data collected successfully.",
                tool_call_id=runtime.tool_call_id,
            )
        ],
    }

    if member_id is not None:
        updates["member_id"] = member_id
    if customer_name is not None:
        updates["customer_name"] = customer_name
    if branch is not None:
        updates["branch"] = branch
    if membership_status is not None:
        updates["membership_status"] = membership_status
    if freeze_start_date is not None:
        updates["freeze_start_date"] = freeze_start_date
    if freeze_duration_days is not None:
        updates["freeze_duration_days"] = freeze_duration_days
    if payment_reference is not None:
        updates["payment_reference"] = payment_reference
    if payment_date is not None:
        updates["payment_date"] = payment_date
    if class_name is not None:
        updates["class_name"] = class_name
    if class_date is not None:
        updates["class_date"] = class_date
    if locker_number is not None:
        updates["locker_number"] = locker_number

    return Command(update=updates)


# -----------------------------------------------------------------------------
# Action tools
# -----------------------------------------------------------------------------
@tool
def process_freeze_membership(
    member_id: str,
    freeze_start_date: str,
    freeze_duration_days: int,
) -> str:
    """Process a membership freeze request."""
    return (
        f"Freeze request submitted successfully for member {member_id}. "
        f"Start date: {freeze_start_date}. "
        f"Duration: {freeze_duration_days} days. "
        f"Reference: FRZ-2026-001."
    )


@tool
def open_billing_case(
    member_id: str,
    payment_reference: str,
    payment_date: str,
) -> str:
    """Open a double-charge billing case."""
    return (
        f"Billing case created for member {member_id}. "
        f"Payment reference: {payment_reference}. "
        f"Payment date: {payment_date}. "
        f"Case ID: BILL-2026-014. "
        f"Expected SLA: 3 business days."
    )


@tool
def troubleshoot_qr_access(
    member_id: str,
    branch: str,
) -> str:
    """Troubleshoot QR access issues."""
    return (
        f"QR access troubleshooting completed for member {member_id} at {branch}. "
        f"Suggested first step: re-login to the app and refresh QR code. "
        f"If the issue continues, ticket ID: QR-2026-021."
    )


@tool
def resolve_class_booking_issue(
    member_id: str,
    class_name: str,
    class_date: str,
) -> str:
    """Handle class booking support issues."""
    return (
        f"Class booking issue recorded for member {member_id}. "
        f"Class: {class_name}. Date: {class_date}. "
        f"Case ID: CLS-2026-009."
    )


@tool
def recover_locker_code(
    member_id: str,
    branch: str,
    locker_number: str,
) -> str:
    """Recover or reset locker code."""
    return (
        f"Locker code support request created for member {member_id} "
        f"at branch {branch}, locker {locker_number}. "
        f"Reference: LCK-2026-003."
    )


@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the case to a human customer service representative."""
    return f"Escalating to human support. Reason: {reason}"


# -----------------------------------------------------------------------------
# Prompts
# -----------------------------------------------------------------------------
ISSUE_CLASSIFIER_PROMPT = """
You are an AI Gym customer service agent.

CURRENT STEP: Issue classification

At this step, you need to:
1. Understand the customer's request or problem
2. Classify it into exactly one of the following:
   - freeze_membership
   - double_charge
   - qr_access_issue
   - class_booking_issue
   - locker_code_issue
   - unknown
3. Use record_issue_type to save the issue type and move to the next step

Be short, clear, and helpful.
If the issue is unclear, ask one clarifying question only if necessary.
"""

CUSTOMER_DATA_COLLECTOR_PROMPT = """
You are an AI Gym customer service agent.

CURRENT STEP: Customer data collection
KNOWN ISSUE TYPE: {issue_type}

Known state:
- member_id: {member_id}
- customer_name: {customer_name}
- branch: {branch}
- membership_status: {membership_status}
- freeze_start_date: {freeze_start_date}
- freeze_duration_days: {freeze_duration_days}
- payment_reference: {payment_reference}
- payment_date: {payment_date}
- class_name: {class_name}
- class_date: {class_date}
- locker_number: {locker_number}

At this step, you need to collect only the necessary missing data for the issue type.

Rules by issue type:

1. freeze_membership
   Required:
   - member_id
   - freeze_start_date
   - freeze_duration_days

2. double_charge
   Required:
   - member_id
   - payment_reference
   - payment_date

3. qr_access_issue
   Required:
   - member_id
   - branch

4. class_booking_issue
   Required:
   - member_id
   - class_name
   - class_date

5. locker_code_issue
   Required:
   - member_id
   - branch
   - locker_number

When enough data is available, use collect_customer_data.
Ask only for missing fields. Do not ask for already known fields.
"""

ISSUE_RESOLUTION_SPECIALIST_PROMPT = """
You are an AI Gym customer service agent.

CURRENT STEP: Issue resolution

Customer state:
- issue_type: {issue_type}
- member_id: {member_id}
- customer_name: {customer_name}
- branch: {branch}
- membership_status: {membership_status}
- freeze_start_date: {freeze_start_date}
- freeze_duration_days: {freeze_duration_days}
- payment_reference: {payment_reference}
- payment_date: {payment_date}
- class_name: {class_name}
- class_date: {class_date}
- locker_number: {locker_number}

At this step, resolve the issue using the correct tool:

- freeze_membership -> process_freeze_membership
- double_charge -> open_billing_case
- qr_access_issue -> troubleshoot_qr_access
- class_booking_issue -> resolve_class_booking_issue
- locker_code_issue -> recover_locker_code
- unknown -> escalate_to_human

If essential information is still missing, ask for it clearly.
If the issue is unsupported or unclear, escalate to human.
"""


# -----------------------------------------------------------------------------
# Step configuration
# -----------------------------------------------------------------------------
STEP_CONFIG = {
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": [],
    },
    "customer_data_collector": {
        "prompt": CUSTOMER_DATA_COLLECTOR_PROMPT,
        "tools": [collect_customer_data],
        "requires": ["issue_type"],
    },
    "issue_resolution_specialist": {
        "prompt": ISSUE_RESOLUTION_SPECIALIST_PROMPT,
        "tools": [
            process_freeze_membership,
            open_billing_case,
            troubleshoot_qr_access,
            resolve_class_booking_issue,
            recover_locker_code,
            escalate_to_human,
        ],
        "requires": ["issue_type"],
    },
}


# -----------------------------------------------------------------------------
# Middleware
# -----------------------------------------------------------------------------
@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Configure prompt and tools based on current workflow step."""

    current_step = request.state.get("current_step", "issue_classifier")
    step_config = STEP_CONFIG[current_step]

    # Validate required state
    for key in step_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"{key} must be set before reaching {current_step}")

    # Create safe prompt formatting values
    prompt_values = {
        "issue_type": request.state.get("issue_type"),
        "member_id": request.state.get("member_id"),
        "customer_name": request.state.get("customer_name"),
        "branch": request.state.get("branch"),
        "membership_status": request.state.get("membership_status"),
        "freeze_start_date": request.state.get("freeze_start_date"),
        "freeze_duration_days": request.state.get("freeze_duration_days"),
        "payment_reference": request.state.get("payment_reference"),
        "payment_date": request.state.get("payment_date"),
        "class_name": request.state.get("class_name"),
        "class_date": request.state.get("class_date"),
        "locker_number": request.state.get("locker_number"),
    }

    system_prompt = step_config["prompt"].format(**prompt_values)

    request = request.override(
        system_prompt=system_prompt,
        tools=step_config["tools"],
    )

    return handler(request)


# -----------------------------------------------------------------------------
# All tools
# -----------------------------------------------------------------------------
all_tools = [
    record_issue_type,
    collect_customer_data,
    process_freeze_membership,
    open_billing_case,
    troubleshoot_qr_access,
    resolve_class_booking_issue,
    recover_locker_code,
    escalate_to_human,
]


# -----------------------------------------------------------------------------
# Create agent
# -----------------------------------------------------------------------------
agent = create_agent(
    model_nemotron3_nano, # Changed 'model' to 'model_nemotron3_nano' to use the defined model
    tools=all_tools,
    state_schema=GymSupportState,
    middleware=[
        apply_step_config,
        SummarizationMiddleware(
            model="gpt-4.1-mini",
            trigger=("tokens", 4000),
            keep=("messages", 10),
        ),
    ],
    checkpointer=InMemorySaver(),
)


# -----------------------------------------------------------------------------
# Test workflow
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    thread_id = str(uuid7())
    config = {"configurable": {"thread_id": thread_id}}

    # Example 1: freeze membership
    result = agent.invoke(
        {
            "messages": [
                HumanMessage("Hi, I want to freeze my membership for one month starting next week.")
            ]
        },
        config,
    )

    result = agent.invoke(
        {
            "messages": [
                HumanMessage("My member ID is M1029 and start date is 2026-04-20.")
            ]
        },
        config,
    )

    result = agent.invoke(
        {
            "messages": [
                HumanMessage("Please freeze it for 30 days.")
            ]
        },
        config,
    )

    for msg in result["messages"]:
        msg.pretty_print()

================================ Human Message =================================

Hi, I want to freeze my membership for one month starting next week.
================================== Ai Message ==================================
Tool Calls:
  record_issue_type (call_ac62af40293e4163851574ae)
 Call ID: call_ac62af40293e4163851574ae
  Args:
    issue_type: freeze_membership
================================= Tool Message =================================
Name: record_issue_type

Issue type recorded as: freeze_membership
================================== Ai Message ==================================

Sure thing! Toset up the freeze, I’ll need a few details:

1. Your **member ID**  
2. The exact **start date** you’d like the freeze to begin (e.g., 2024‑05‑22)  
3. The **duration** in days (you mentioned one month—should we use 30 days?)

Could you provide those for me?
================================ Human Message =================================

My member ID is M1029 and start date 